<a href="https://colab.research.google.com/github/Harishchand83077/SustAInify/blob/main/SustAInify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [2]:
path="/content/processed_data.csv"
df=pd.read_csv(path)

In [3]:
df = df.dropna()
data = df['Average Prices (EUR/MWh)'].values.reshape(-1, 1)

In [4]:
df

,Date,Average Prices (EUR/MWh)
0,2010-07-21,47.803208
1,2010-07-22,58.061250
2,2010-07-23,46.146042
3,2010-07-24,43.593292
4,2010-07-25,19.148125
...,...,...
2717,2017-12-27,30.209583
2718,2017-12-28,33.220417
2719,2017-12-29,23.760000
2720,2017-12-30,24.729167


In [5]:
# df = df.dropna(axis=0)
# plt.figure(figsize=(24, 30))
# plt.plot(df['Average Prices (EUR/MWh)'], marker='o')
# plt.title('Average Prices (EUR/MWh) vs Time Steps')
# plt.xlabel('Time Steps')
# plt.ylabel('Average Prices (EUR/MWh)')
# plt.show()

In [6]:
# Normalize data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)

# Split data into train and test
train_size = int(len(scaled_data) * 0.8)
train_data, test_data = scaled_data[:train_size], scaled_data[train_size:]

In [7]:
# # Find the threshold values for bottom 1% and top 1%
# data = pd.to_numeric(df['Average Prices (EUR/MWh)'], errors='coerce')

# # Find the threshold values for bottom 1% and top 1%
# bottom_threshold = data.quantile(0.05)
# top_threshold = data.quantile(0.95)

# # Identify the indices that need to be replaced
# indices_to_replace = (data < bottom_threshold) | (data > top_threshold)

# # Iterate over the indices to replace values with the average of 5 values before and after
# for index in indices_to_replace.index:
#     if indices_to_replace[index]:
#         idx = df.index.get_loc(index)
#         # Calculate the average of 5 values before and after
#         avg_value = data.iloc[max(0, idx - 5):min(len(df), idx + 6)].mean()
#         data.at[index] = avg_value
# print(len(data))

In [8]:
# plt.figure(figsize=(24, 30))
# plt.plot(data, marker='o')
# plt.title('Average Prices (EUR/MWh) vs Time Steps')
# plt.xlabel('Time Steps')
# plt.ylabel('Average Prices (EUR/MWh)')
# plt.show()

In [9]:
# window_size = 7
# data = data.rolling(window=window_size).mean()
# plt.figure(figsize=(24, 30))
# plt.plot(data, marker='o')
# plt.title('Average Prices (EUR/MWh) vs Time Steps')
# plt.xlabel('Time Steps')
# plt.ylabel('Average Prices (EUR/MWh)')
# plt.show()

In [10]:
# Train ARIMA model
arima_order = (5,1,0)
arima_model = ARIMA(train_data, order=arima_order)
arima_fit = arima_model.fit()
arima_pred = arima_fit.fittedvalues
residuals = train_data.flatten() - arima_pred

In [11]:
!pip install --upgrade scikit-learn xgboost

  Using cached scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
Using cached scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.5 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.0
    Uninstalling scikit-learn-1.2.0:
      Successfully uninstalled scikit-learn-1.2.0


In [12]:
!pip install scikit-learn==1.2.0 xgboost

  Using cached scikit_learn-1.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (9.5 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 1.35.0 requires scikit-learn>=1.2.2, but you have scikit-learn 1.2.0 which is incompatible.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.0 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.0 which is incompatible.


In [13]:
# Train XGBoost on ARIMA residuals
X_train = np.arange(len(residuals)).reshape(-1, 1)
y_train = residuals
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', learning_rate=0.1, max_depth=3, n_estimators=100)
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [14]:
# Predict future values using ARIMA-XGBoost
def predict_arima_xgboost(arima_fit, xgb_model, steps):
    arima_forecast = arima_fit.forecast(steps=steps)
    X_test = np.arange(len(arima_forecast)).reshape(-1, 1)
    xgb_correction = xgb_model.predict(X_test)
    return arima_forecast + xgb_correction

In [15]:
# Predict daily average electricity prices for 2018
days_2018 = 365
predictions = predict_arima_xgboost(arima_fit, xgb_model, days_2018)
predictions = scaler.inverse_transform(predictions.reshape(-1, 1))

# Save predictions to CSV
predicted_dates = pd.date_range(start='2018-01-01', periods=days_2018, freq='D')
predicted_df = pd.DataFrame({'Date': predicted_dates, 'Predicted Average Price (EUR/MWh)': predictions.flatten()})
predicted_df.to_csv('Predicted_Electricity_Prices_2018.csv', index=False)

In [16]:
# Optimization of electricity procurement costs and environmental impact
def objective(vars):
    Q_Grid, Q_Exchange = vars
    cost = (Q_Grid * 57.62) + (Q_Exchange * predicted_df.iloc[0, 1])  # Cost based on 1st Jan 2018
    return cost

def constraint(vars):
    Q_Grid, Q_Exchange = vars
    return 1200 - (Q_Grid + Q_Exchange + 150)  # Ensuring total demand is met

def renewable_constraint(vars):
    Q_Grid, Q_Exchange = vars
    renewable_percentage = ((0.15 * Q_Grid) + (0.05 * Q_Exchange) + 150) / 1200
    return renewable_percentage - 0.2

In [18]:
# Initial guess and bounds
from scipy.optimize import minimize # Import the minimize function

x0 = [500, 500]
bounds = [(0, 1200), (0, 1200)]
constraints = ({'type': 'eq', 'fun': constraint}, {'type': 'ineq', 'fun': renewable_constraint})
result = minimize(objective, x0, bounds=bounds, constraints=constraints)

Q_Grid_opt, Q_Exchange_opt = result.x
renewable_percentage_opt = ((0.15 * Q_Grid_opt) + (0.05 * Q_Exchange_opt) + 150) / 1200 * 100

# Print optimized results
print("Optimized Percentage of Renewable Electricity:", renewable_percentage_opt)
print("Optimized Quantity from Grid (Q_Grid):", Q_Grid_opt)
print("Optimized Quantity from Power Exchange (Q_Exchange):", Q_Exchange_opt)

Optimized Percentage of Renewable Electricity: 19.999999999937934
Optimized Quantity from Grid (Q_Grid): 374.9999999925522
Optimized Quantity from Power Exchange (Q_Exchange): 675.0000000074477
